# TP ChromaDB - corrigé
    
Ce notebook guide vos étapes pour découvrir et pratiquer ChromaDB (base de données vectorielle).
Il est structuré en 4 séquences.
    
**Prérequis** : Python 3.13+
    
> Astuce : Exécutez chaque cellule dans l’ordre. Documentez vos observations dans les zones prévues.


---
## 0) Préparation de l'environnement

Objectifs :
- Installer les dépendances nécessaires : `chromadb`
- Vérifier les versions.


In [ ]:
# !pip install chromadb
import chromadb

---
## 1) Découverte et premier index

**Objectifs**
- Comprendre le concept d'embeddings et de recherche de similarité.
- Créer un client ChromaDB, une collection, ajouter quelques documents, interroger.


In [15]:
# Création d'un client avec une base de données persistances
client = chromadb.PersistentClient()

In [16]:
# Création d'une collection (<=> table en sgbdr traditionnel)
collection = client.get_or_create_collection(name="LaReunion")

In [17]:
# Déclaration des textes à ajouter à notre collection
doc1 = "La préfecture de La Réunion se situe à Saint-Denis."
doc2 = "ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique."
doc3 = "Le piton de la Fournaise est l'un des volcans les plus actifs du monde."

In [18]:
# Ajout des metadonnées et des identifiants uniques
docs = [doc1, doc2, doc3]
ids = ["id1", "id2", "id3"]
metadatas = [
    {"source": "wiki", "categorie": "géographie"},
    {"source": "doc", "categorie": "technique"},
    {"source": "wiki", "categorie": "géographie"}
]

In [19]:
# Ajout des données dans la collection
collection.add(
    documents=docs,
    ids=ids,
    metadatas=metadatas
)
print("Documents ajoutés. Count =", collection.count())

Documents ajoutés. Count = 3


In [20]:
# Recherche de similitude
query_text = "Où se trouve la préfecture de l'île ?"
results = collection.query(
    query_texts=[query_text],
    n_results=2
)

In [21]:
results

{'ids': [['id1', 'id3']],
 'embeddings': None,
 'documents': [['La préfecture de La Réunion se situe à Saint-Denis.',
   "Le piton de la Fournaise est l'un des volcans les plus actifs du monde."]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'wiki', 'categorie': 'géographie'},
   {'source': 'wiki', 'categorie': 'géographie'}]],
 'distances': [[0.8978041410446167, 1.2525721788406372]]}

In [22]:
collection.peek()

{'ids': ['id1', 'id2', 'id3'],
 'embeddings': array([[-0.04831972,  0.01283783, -0.02138843, ...,  0.00970293,
         -0.0125578 , -0.0198486 ],
        [-0.10025828, -0.00313545, -0.04016039, ...,  0.10073496,
          0.05870798, -0.07088163],
        [-0.02009018,  0.01328702, -0.02889018, ...,  0.03941492,
          0.06404133,  0.04178637]], shape=(3, 384)),
 'documents': ['La préfecture de La Réunion se situe à Saint-Denis.',
  'ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique.',
  "Le piton de la Fournaise est l'un des volcans les plus actifs du monde."],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'categorie': 'géographie', 'source': 'wiki'},
  {'categorie': 'technique', 'source': 'doc'},
  {'source': 'wiki', 'categorie': 'géographie'}]}

> **Observation (à compléter)**  
- Qu'ai-je observé sur les résultats (ordre, distances, ids) ?  
- À quoi servent les embeddings ?
- Quel est l'embedder par défaut utilisé par chromadb ?


---
## 2) Métadonnées, filtrage, mise à jour et suppression

**Objectifs**
- Ajouter un jeu de documents plus riche (10–15+).
- Utiliser des métadonnées et effectuer des recherches avec filtres.
- Mettre à jour et supprimer des entrées.


In [23]:
# TODO: Créez ou chargez un corpus de 10–15 documents
# Exemple : fabriquez un corpus simple ici. Vous pouvez remplacer par vos propres textes.
more_docs = [
    "Les plages de l'ouest de La Réunion sont protégées par une barrière de corail.",
    "Les métadonnées permettent de filtrer des recherches sémantiques dans ChromaDB.",
    "La recherche sémantique se base sur la similarité de sens, pas uniquement sur les mots exacts.",
    "Le Piton des Neiges est le point culminant de l'île de La Réunion.",
    "Les baleines sont visibles à certaines périodes de l'année au large de La Réunion.",
    "ChromaDB supporte la persistance via DuckDB/Parquet par défaut.",
    "On peut filtrer par 'categorie' ou 'source' pour affiner les résultats.",
    "Mettre à jour un document permet de corriger ou d'améliorer les informations.",
    "Supprimer un document est nécessaire si l'entrée devient obsolète.",
    "L'intégration avec des LLMs permet d'augmenter les capacités de QA."
]
more_ids = [f"mid{i+1}" for i in range(len(more_docs))]
more_metadatas = [
    {"source": "guide", "categorie": "tourisme"},
    {"source": "note", "categorie": "technique"},
    {"source": "note", "categorie": "technique"},
    {"source": "guide", "categorie": "geographie"},
    {"source": "guide", "categorie": "nature"},
    {"source": "doc", "categorie": "technique"},
    {"source": "doc", "categorie": "technique"},
    {"source": "doc", "categorie": "technique"},
    {"source": "doc", "categorie": "technique"},
    {"source": "doc", "categorie": "llm"}
]

In [24]:
# Ajouter de nouveaux documents
collection.add(documents=more_docs, ids=more_ids, metadatas=more_metadatas)
print("Nouveaux documents ajoutés. Count =", collection.count())
print("Aperçu (peek) :")
collection.peek()

Nouveaux documents ajoutés. Count = 13
Aperçu (peek) :


{'ids': ['id1',
  'id2',
  'id3',
  'mid1',
  'mid2',
  'mid3',
  'mid4',
  'mid5',
  'mid6',
  'mid7'],
 'embeddings': array([[-0.04831972,  0.01283783, -0.02138843, ...,  0.00970293,
         -0.0125578 , -0.0198486 ],
        [-0.10025828, -0.00313545, -0.04016039, ...,  0.10073496,
          0.05870798, -0.07088163],
        [-0.02009018,  0.01328702, -0.02889018, ...,  0.03941492,
          0.06404133,  0.04178637],
        ...,
        [-0.02808154,  0.03215837,  0.03380241, ...,  0.11710104,
          0.02866336, -0.01025276],
        [-0.09746863, -0.01266403, -0.10858261, ...,  0.03414119,
          0.11910061,  0.01831229],
        [ 0.01371803,  0.04820116, -0.02823632, ...,  0.07246908,
          0.04974221, -0.01123135]], shape=(10, 384)),
 'documents': ['La préfecture de La Réunion se situe à Saint-Denis.',
  'ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique.',
  "Le piton de la Fournaise est l'un des volcans les plus actifs du m

In [25]:
collection.peek()['embeddings'][0]

array([-4.83197235e-02,  1.28378263e-02, -2.13884283e-02, -4.73451391e-02,
       -2.46241484e-02,  4.11578789e-02, -9.99689102e-02,  3.17227989e-02,
        1.03429724e-02, -3.79660539e-02,  9.15957317e-02, -6.54921457e-02,
        3.44646238e-02, -6.00603484e-02,  4.03489172e-02, -1.01528645e-01,
       -7.34445974e-02,  8.35025534e-02,  9.10082981e-02,  5.07145934e-02,
       -4.13519740e-02,  3.33518628e-03, -3.06899734e-02, -1.73310889e-03,
        6.12043068e-02,  4.50613461e-02,  4.43272991e-03, -2.44674478e-02,
       -1.37699554e-02,  2.89325719e-03,  5.96546717e-02,  2.98015177e-02,
        3.32656577e-02, -2.41413433e-02,  1.89461708e-02,  2.53496226e-02,
        5.25622629e-03, -5.06523736e-02, -2.54763868e-02,  5.64819351e-02,
       -1.38229191e-01,  1.54493004e-02,  3.78424465e-03, -5.09139858e-02,
       -3.87608483e-02,  1.20702134e-02, -2.19930038e-02,  5.89429736e-02,
       -1.29922824e-02, -1.09093815e-01,  6.12150170e-02, -4.30420376e-02,
        5.04386127e-02,  

In [26]:
# Recherche sans filtre
q = "Comment affiner une recherche sémantique ?"
res_no_filter = collection.query(query_texts=[q], n_results=5)
res_no_filter

{'ids': [['mid3', 'mid2', 'id2', 'mid7', 'mid8']],
 'embeddings': None,
 'documents': [['La recherche sémantique se base sur la similarité de sens, pas uniquement sur les mots exacts.',
   'Les métadonnées permettent de filtrer des recherches sémantiques dans ChromaDB.',
   'ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique.',
   "On peut filtrer par 'categorie' ou 'source' pour affiner les résultats.",
   "Mettre à jour un document permet de corriger ou d'améliorer les informations."]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'note', 'categorie': 'technique'},
   {'source': 'note', 'categorie': 'technique'},
   {'categorie': 'technique', 'source': 'doc'},
   {'categorie': 'technique', 'source': 'doc'},
   {'categorie': 'technique', 'source': 'doc'}]],
 'distances': [[0.7369275093078613,
   1.044176459312439,
   1.0793112516403198,
   1.0936464071273804,
   1.250197887420654

In [27]:
# Recherche avec filtre métadonnée (ex.: seulement 'technique')
res_with_filter = collection.query(
    query_texts=["Comment persister mes données ?"],
    n_results=5,
    where={"categorie": "technique"}  # Filtre exact
)
res_with_filter

{'ids': [['mid6', 'mid9', 'mid3', 'mid8', 'mid7']],
 'embeddings': None,
 'documents': [['ChromaDB supporte la persistance via DuckDB/Parquet par défaut.',
   "Supprimer un document est nécessaire si l'entrée devient obsolète.",
   'La recherche sémantique se base sur la similarité de sens, pas uniquement sur les mots exacts.',
   "Mettre à jour un document permet de corriger ou d'améliorer les informations.",
   "On peut filtrer par 'categorie' ou 'source' pour affiner les résultats."]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'doc', 'categorie': 'technique'},
   {'source': 'doc', 'categorie': 'technique'},
   {'source': 'note', 'categorie': 'technique'},
   {'source': 'doc', 'categorie': 'technique'},
   {'categorie': 'technique', 'source': 'doc'}]],
 'distances': [[1.678590178489685,
   1.6789114475250244,
   1.7071796655654907,
   1.7175381183624268,
   1.8104771375656128]]}

In [46]:
# Mise à jour d'un document (update)
# Exemple : on met à jour le texte de 'mdoc2'
collection.update(
    ids=["mid2"],
    documents=["Les métadonnées sont essentielles pour filtrer précisément les recherches dans ChromaDB."]
)

In [47]:
collection.get("mid2")

{'ids': ['mid2'],
 'embeddings': None,
 'documents': ['Les métadonnées sont essentielles pour filtrer précisément les recherches dans ChromaDB.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'categorie': 'technique', 'source': 'note'}]}

In [29]:
collection.peek()

{'ids': ['id1',
  'id2',
  'id3',
  'mid1',
  'mid2',
  'mid3',
  'mid4',
  'mid5',
  'mid6',
  'mid7'],
 'embeddings': array([[-0.04831972,  0.01283783, -0.02138843, ...,  0.00970293,
         -0.0125578 , -0.0198486 ],
        [-0.10025828, -0.00313545, -0.04016039, ...,  0.10073496,
          0.05870798, -0.07088163],
        [-0.02009018,  0.01328702, -0.02889018, ...,  0.03941492,
          0.06404133,  0.04178637],
        ...,
        [-0.02808154,  0.03215837,  0.03380241, ...,  0.11710104,
          0.02866336, -0.01025276],
        [-0.09746863, -0.01266403, -0.10858261, ...,  0.03414119,
          0.11910061,  0.01831229],
        [ 0.01371803,  0.04820116, -0.02823632, ...,  0.07246908,
          0.04974221, -0.01123135]], shape=(10, 384)),
 'documents': ['La préfecture de La Réunion se situe à Saint-Denis.',
  'ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique.',
  "Le piton de la Fournaise est l'un des volcans les plus actifs du m

In [30]:
# Suppression d'un document
collection.delete(ids=["mdoc9"])
print("Après update/delete, count =", collection.count())

Après update/delete, count = 13


> **Synthèse (à compléter)**  
- Dans quels cas les filtres par métadonnées m'ont aidé ?  
- Quel impact des mises à jour/suppressions sur les résultats ?


---
## 3) Cas pratique : recherche sémantique sur mini‑corpus

**Objectifs**
- Construire un corpus sur un thème choisi.
- Lancer plusieurs requêtes et analyser la pertinence.
- (Bonus) Chunking si les documents sont longs.

**Livrable**
- Notebook avec corpus + requêtes + analyse.


In [ ]:
# TODO: Remplacez par votre thème/corpus (ex.: FAQ interne, fiches produits, extraits d'articles)
theme_docs = [
    # Ajoutez ici vos textes liés à un même thème
]
theme_ids = []
theme_metas = []

# Exemple d'ajout (décommentez quand prêts)
# collection_theme = client.get_or_create_collection(name="tp_theme")
# collection_theme.add(documents=theme_docs, ids=theme_ids, metadatas=theme_metas)
# print("Collection 'tp_theme' prête avec", collection_theme.count(), "documents.")

In [ ]:
# TODO: Définissez 3–5 requêtes utilisateur et interrogez la collection
# queries = ["...", "...", "..."]
# for q in queries:
#     res = collection_theme.query(query_texts=[q], n_results=3)
#     print("\nQuery:", q)
#     print(json.dumps(res, indent=2, ensure_ascii=False))

> **Analyse (à compléter)**  
Pour chaque requête :
- Les documents renvoyés sont‑ils pertinents ? Pourquoi ?  
- Les distances/scores reflètent‑ils votre intuition ?


---
## 4) Extension : mini‑application & bonnes pratiques

**Objectifs**
- Construire une petite interface d'interrogation en console.
- Persister les données et discuter des bonnes pratiques.

**Livrable**
- Script/Notebook de mini‑app + fiche « bonnes pratiques » (1 page).


In [13]:
# Mini‑app console (simple)
COLLECTION_NAME = "LaReunion"
# Utilise la collection de base. Tapez 'quit' pour sortir.
print("Mini‑app de recherche (collection:", COLLECTION_NAME, ")")
while True:
    user_q = input("Votre question (ou 'quit'): ").strip()
    if user_q.lower() in {"quit", "exit"}:
        print("Bye!")
        break
    res = collection.query(query_texts=[user_q], n_results=3)
    # Affichage synthétique
    for i, doc_id in enumerate(res.get("ids", [[]])[0]):
        doc = res.get("documents", [[]])[0][i]
        meta = res.get("metadatas", [[]])[0][i]
        dist = res.get("distances", [[]])[0][i] if "distances" in res else None
        print(f"[{i+1}] id={doc_id} dist={dist} meta={meta}\n-> {doc[:180]}...\n")

Mini‑app de recherche (collection: LaReunion )
[1] id=id1 dist=0.7552140951156616 meta={'categorie': 'géographie', 'source': 'wiki'}
-> La préfecture de La Réunion se situe à Saint-Denis....

[2] id=id3 dist=1.1707763671875 meta={'source': 'wiki', 'categorie': 'géographie'}
-> Le piton de la Fournaise est l'un des volcans les plus actifs du monde....

[3] id=id2 dist=1.6281790733337402 meta={'categorie': 'technique', 'source': 'doc'}
-> ChromaDB est une base de données vectorielle simple à utiliser pour la recherche sémantique....

[1] id=id3 dist=1.9127357006072998 meta={'source': 'wiki', 'categorie': 'géographie'}
-> Le piton de la Fournaise est l'un des volcans les plus actifs du monde....

[2] id=id1 dist=1.9494764804840088 meta={'source': 'wiki', 'categorie': 'géographie'}
-> La préfecture de La Réunion se situe à Saint-Denis....

[3] id=id2 dist=2.0183773040771484 meta={'source': 'doc', 'categorie': 'technique'}
-> ChromaDB est une base de données vectorielle simple à utiliser po

---
## Annexes

### Utilisation d'un autre embedder

Par défaut chromadb fait appel à l'embedder `sentence-transformers/all-MiniLM-L6-v2`.  
cf. https://docs.trychroma.com/docs/embeddings/embedding-functions#default-all-minilm-l6-v2

In [51]:
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
ollama_ef = OllamaEmbeddingFunction(model_name="nomic-embed-text:latest")

In [ ]:
# # Création des embeddings
# docs_embeddings = ollama_ef(docs)
# print(docs_embeddings)
#
# # Création d'une nouvelle collection avec les nouveaux embeddings 
# collection2 = client.get_or_create_collection(name="LaReunion2")
# collection2.add(
#     embeddings = docs_embeddings,
#     documents = docs,
#     metadatas = metadatas,
#     ids = ids
# )

In [49]:
# Alternative avec embedder intégré dans la collection
collection2 = client.get_or_create_collection(name="LaReunion2", embedding_function=ollama_ef)
collection2.add(
    documents = docs,
    metadatas = metadatas,
    ids = ids
)

In [50]:
# Test d'une requête similaire à la nouvelle collection
results2 = collection2.query(
    query_texts=[query_text],
    n_results=2
)
print(results2)

{'ids': [['id1', 'id3']], 'embeddings': None, 'documents': [['La préfecture de La Réunion se situe à Saint-Denis.', "Le piton de la Fournaise est l'un des volcans les plus actifs du monde."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'categorie': 'géographie', 'source': 'wiki'}, {'source': 'wiki', 'categorie': 'géographie'}]], 'distances': [[0.2588871121406555, 0.42506152391433716]]}


### Bonnes pratiques (à discuter / compléter)
- Utiliser `chromadb.PersistentClient()` pour conserver l'index entre sessions.
- Choisir un modèle d'embeddings adapté (local vs API) ; garder la cohérence entre index et requêtes.
- Pré‑traiter les textes (nettoyage, normalisation, segmentation en *chunks* si longs).
- Ajouter des métadonnées utiles dès l’ingestion (source, date, catégorie, langue).
- Tester et itérer : qualifier la pertinence via des jeux de requêtes réels.
- Surveiller le volume de données et la latence ; adapter le backend si besoin.


---
## Références utiles (à consulter)
- Documentation Chroma (Getting Started, API) : https://docs.trychroma.com/
- Répertoire GitHub : https://github.com/chroma-core/chroma
- Concepts embeddings (Sentence-Transformers) : https://www.sbert.net/
    
> **Note** : Vérifiez la version de `chromadb` et les éventuels changements d'API.
